# Cellule 1: Setup et imports

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import json
import zipfile
import requests
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

# Cellule 2: Téléchargement des données

In [6]:
# Créer le dossier s'il n'existe pas
os.makedirs('medical_bios_data', exist_ok=True)

# URL du dataset
url = "https://huggingface.co/datasets/coastalcph/medical-bios/resolve/main/bios.zip"
zip_path = 'medical_bios_data/bios.zip'

print("🔄 Téléchargement du dataset...")
try:
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    total_size = int(response.headers.get('content-length', 0))
    with open(zip_path, 'wb') as f:
        downloaded = 0
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)
                if total_size > 0:
                    progress = (downloaded / total_size) * 100
                    print(f"\r📥 Téléchargement: {progress:.1f}%", end="", flush=True)
    
    print(f"\n✅ Téléchargement terminé: {Path(zip_path).stat().st_size / (1024*1024):.2f} MB")
    
except Exception as e:
    print(f"❌ Erreur de téléchargement: {e}")

🔄 Téléchargement du dataset...
📥 Téléchargement: 100.0%
✅ Téléchargement terminé: 1.82 MB


# Cellule 3: Extraction automatique

In [7]:
def extract_dataset():
    """Extraction automatique du dataset"""
    zip_path = 'medical_bios_data/bios.zip'
    extract_path = 'medical_bios_data/'
    
    if not os.path.exists(zip_path):
        print("❌ Fichier zip non trouvé. Veuillez d'abord télécharger les données.")
        return False
        
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
            print("✅ Extraction terminée!")
            
            # Lister les fichiers extraits
            print("\n📋 Fichiers extraits:")
            for file_info in zip_ref.infolist():
                if file_info.filename.endswith('.jsonl'):
                    print(f"  📄 {file_info.filename} ({file_info.file_size:,} bytes)")
        return True
        
    except Exception as e:
        print(f"❌ Erreur d'extraction: {e}")
        return False

# Exécuter l'extraction
extraction_success = extract_dataset()

✅ Extraction terminée!

📋 Fichiers extraits:
  📄 test_rationales.jsonl (300,522 bytes)
  📄 __MACOSX/._test_rationales.jsonl (176 bytes)
  📄 test.jsonl (946,345 bytes)
  📄 __MACOSX/._test.jsonl (275 bytes)
  📄 train.jsonl (7,595,575 bytes)
  📄 __MACOSX/._train.jsonl (219 bytes)
  📄 validation.jsonl (938,592 bytes)
  📄 __MACOSX/._validation.jsonl (219 bytes)


# Cellule 4: Chargement des données

In [11]:
def load_jsonl(file_path):
    """Charger un fichier JSONL en liste de dictionnaires"""
    if not os.path.exists(file_path):
        print(f"❌ Fichier non trouvé: {file_path}")
        return []
        
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

if extraction_success:
    # Charger tous les datasets
    print("\n🔄 Chargement des datasets...")
    
    train_set = load_jsonl('medical_bios_data/train.jsonl')
    val_set = load_jsonl('medical_bios_data/validation.jsonl') 
    test_set = load_jsonl('medical_bios_data/test.jsonl')
    
    print(f"✅ Train: {len(train_set)} exemples")
    print(f"✅ Validation: {len(val_set)} exemples") 
    print(f"✅ Test: {len(test_set)} exemples")
    
    # Convertir en DataFrames
    df_train = pd.DataFrame(train_set)
    df_val = pd.DataFrame(val_set)
    df_test = pd.DataFrame(test_set)
    
    print(f"\n📊 Colonnes disponibles: {list(df_train.columns)}")
    print(f"📋 Exemple de données:")
    print(df_train.head(2))


🔄 Chargement des datasets...
✅ Train: 8000 exemples
✅ Validation: 1000 exemples
✅ Test: 1000 exemples

📊 Colonnes disponibles: ['full_text', 'text', 'title', 'gender', 'date']
📋 Exemple de données:
                                           full_text  \
0  Dr. Vikram Prasad is an experienced Dentist in...   
1  Dr. Randy Clark is an orthopedic surgeon origi...   

                                                text    title gender     date  
0  He has been a practicing Dentist for 20 years....  dentist   Male  2018-09  
1  He was happy to return to this area with his w...  surgeon   Male  2017-39  


# Cellule 5: Analyse exploratoire rapide

In [13]:
if extraction_success:
    print("\n🔍 ANALYSE EXPLORATOIRE")
    print("=" * 30)
    
    # Distribution des genres
    print("👥 Distribution par genre:")
    print(df_train['gender'].value_counts())
    
    # Pourcentages corrigés
    gender_percentages = df_train['gender'].value_counts(normalize=True) * 100
    print("Pourcentages:")
    for gender, pct in gender_percentages.items():
        print(f"  {gender}: {pct:.1f}%")
    
    # Distribution des professions
    print(f"\n🏥 Distribution par profession:")
    print(df_train['title'].value_counts())
    
    # Vérification des données manquantes
    print(f"\n⚠️ Données manquantes:")
    missing_data = df_train.isnull().sum()
    for col, missing in missing_data.items():
        if missing > 0:
            print(f"  {col}: {missing} ({missing/len(df_train)*100:.1f}%)")
    
    if missing_data.sum() == 0:
        print("  ✅ Aucune donnée manquante!")


🔍 ANALYSE EXPLORATOIRE
👥 Distribution par genre:
gender
Female    4290
Male      3710
Name: count, dtype: int64
Pourcentages:
  Female: 53.6%
  Male: 46.4%

🏥 Distribution par profession:
title
psychologist    2200
nurse           1638
dentist         1533
physician       1349
surgeon         1280
Name: count, dtype: int64

⚠️ Données manquantes:
  ✅ Aucune donnée manquante!


# Cellule 6: Préparation simple pour baseline 

In [14]:
if extraction_success:
    print("\n🔧 PRÉPARATION POUR BASELINE SIMPLE")
    print("=" * 40)
    
    # Encoder les labels
    le = LabelEncoder()
    all_titles = list(df_train['title']) + list(df_val['title']) + list(df_test['title'])
    le.fit(all_titles)
    
    # Appliquer l'encodage
    df_train['title_encoded'] = le.transform(df_train['title'])
    df_val['title_encoded'] = le.transform(df_val['title'])
    df_test['title_encoded'] = le.transform(df_test['title'])
    
    print("🏷️ Mapping des labels:")
    for i, profession in enumerate(le.classes_):
        print(f"  {i}: {profession}")
    
    # Sauvegarder le label encoder
    with open('models/label_encoder.pkl', 'wb') as f:
        pickle.dump(le, f)
    print("✅ LabelEncoder sauvegardé")


🔧 PRÉPARATION POUR BASELINE SIMPLE
🏷️ Mapping des labels:
  0: dentist
  1: nurse
  2: physician
  3: psychologist
  4: surgeon
✅ LabelEncoder sauvegardé


# Cellule 7: Baseline avec TF-IDF (plus simple que transformer)

In [15]:
if extraction_success:
    print("\n🤖 BASELINE AVEC TF-IDF")
    print("=" * 30)
    
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.pipeline import Pipeline
    
    # Créer le pipeline TF-IDF + RandomForest
    baseline_pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=5000,
            stop_words='english', 
            lowercase=True,
            ngram_range=(1, 2)
        )),
        ('classifier', RandomForestClassifier(
            n_estimators=100,
            max_depth=20,
            random_state=42,
            n_jobs=-1
        ))
    ])
    
    # Préparer les données
    X_train = df_train['full_text'].fillna('')
    y_train = df_train['title_encoded']
    X_val = df_val['full_text'].fillna('')
    y_val = df_val['title_encoded']
    X_test = df_test['full_text'].fillna('')
    y_test = df_test['title_encoded']
    
    print("🔄 Entraînement du modèle baseline...")
    baseline_pipeline.fit(X_train, y_train)
    print("✅ Entraînement terminé!")
    
    # Prédictions
    train_pred = baseline_pipeline.predict(X_train)
    val_pred = baseline_pipeline.predict(X_val)
    test_pred = baseline_pipeline.predict(X_test)
    
    # Métriques
    train_acc = accuracy_score(y_train, train_pred)
    val_acc = accuracy_score(y_val, val_pred) 
    test_acc = accuracy_score(y_test, test_pred)
    
    print(f"\n📊 RÉSULTATS BASELINE:")
    print(f"  🎯 Accuracy Train: {train_acc:.4f} ({train_acc*100:.2f}%)")
    print(f"  🎯 Accuracy Validation: {val_acc:.4f} ({val_acc*100:.2f}%)")
    print(f"  🎯 Accuracy Test: {test_acc:.4f} ({test_acc*100:.2f}%)")
    
    # Sauvegarder le modèle
    with open('models/baseline_model.pkl', 'wb') as f:
        pickle.dump(baseline_pipeline, f)
    print("✅ Modèle baseline sauvegardé")


🤖 BASELINE AVEC TF-IDF
🔄 Entraînement du modèle baseline...
✅ Entraînement terminé!

📊 RÉSULTATS BASELINE:
  🎯 Accuracy Train: 0.9985 (99.85%)
  🎯 Accuracy Validation: 0.9890 (98.90%)
  🎯 Accuracy Test: 0.9910 (99.10%)
✅ Modèle baseline sauvegardé


# Cellule 8: Analyse de biais de base

In [ ]:
if extraction_success:
    print("\n⚖️ ANALYSE DE BIAIS PRÉLIMINAIRE")
    print("=" * 35)
    
    # Préparer les données de test avec genres
    df_test_results = df_test.copy()
    df_test_results['predicted'] = test_pred
    df_test_results['correct'] = (df_test_results['title_encoded'] == test_pred)
    
    # Accuracy par genre
    print("📊 Accuracy par genre:")
    gender_accuracy = df_test_results.groupby('gender')['correct'].agg(['mean', 'count'])
    for gender in gender_accuracy.index:
        acc = gender_accuracy.loc[gender, 'mean']
        count = gender_accuracy.loc[gender, 'count']
        print(f"  {gender}: {acc:.3f} ({acc*100:.1f}%) - {count} exemples")
    
    # Différence d'accuracy entre genres
    if len(gender_accuracy) >= 2:
        genders = list(gender_accuracy.index)
        acc_diff = abs(gender_accuracy.loc[genders[0], 'mean'] - 
                      gender_accuracy.loc[genders[1], 'mean'])
        print(f"\n⚠️ Écart d'accuracy: {acc_diff:.3f} ({acc_diff*100:.1f} points)")
        
        if acc_diff > 0.05:  # 5% de différence
            print("🚨 BIAIS DÉTECTÉ: Écart significatif entre genres!")
        else:
            print("✅ Écart acceptable entre genres")
    
    # Distribution des prédictions par genre
    print(f"\n📈 Distribution des prédictions par genre:")
    pred_by_gender = pd.crosstab(df_test_results['gender'], 
                                df_test_results['predicted'], 
                                normalize='index') * 100
    print(pred_by_gender.round(1))

print("\n🎉 BASELINE TERMINÉE!")
print("📁 Fichiers générés:")
print("  - label_encoder.pkl")  
print("  - baseline_model.pkl")


⚖️ ANALYSE DE BIAIS PRÉLIMINAIRE
📊 Accuracy par genre:
  Female: 0.989 (98.9%) - 540 exemples
  Male: 0.993 (99.3%) - 460 exemples

⚠️ Écart d'accuracy: 0.005 (0.5 points)
✅ Écart acceptable entre genres

📈 Distribution des prédictions par genre:
predicted     0     1     2     3     4
gender                                 
Female     12.4  32.4  17.8  32.6   4.8
Male       28.3   3.5  15.7  23.9  28.7

🎉 BASELINE TERMINÉE!
📁 Fichiers générés:
  - label_encoder.pkl
  - baseline_model.pkl

🔄 Prochaines étapes: analyser les biais en détail!
